# My Models

Live lists of every model my API keys can use right now, plus balances and a quick way to try any model.

**Run All** to refresh everything. Then copy a model name into a `_mine` notebook's setup cell, for example:

```python
PROVIDER = "openrouter"
SMALL, MEDIUM, LARGE = models(PROVIDER)
MEDIUM = "moonshotai/kimi-k2.6"     # any name from the matching table below
```

`MY_MODELS.md` explains how each lab is wired, and `my_llm.py` holds my default models.

In [ ]:
import os
import time

import pandas as pd
import requests
from IPython.display import display

from my_llm import MODELS, PROVIDERS, client, setup

setup()
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_colwidth", 70)

# Which keys are in .env (values are never printed)
for name, provider in PROVIDERS.items():
    key = provider["key"]
    status = "no key needed" if key is None else ("set" if os.getenv(key) else "MISSING from .env")
    print(f"{name:11} {key or '':20} {status}")

## My defaults

What `models(PROVIDER)` returns in every `_mine` notebook. Change these in `my_llm.py`.

In [ ]:
pd.DataFrame(MODELS, index=["SMALL", "MEDIUM", "LARGE"]).T

## OpenRouter

One key, hundreds of models from every major lab. Prices are US dollars per million tokens.
Use the **model** column exactly as shown (for example `anthropic/claude-sonnet-5.5`).

In [ ]:
def openrouter_models():
    data = requests.get("https://openrouter.ai/api/v1/models", timeout=30).json()["data"]
    rows = []
    for m in data:
        params = m.get("supported_parameters") or []
        price = m.get("pricing") or {}
        rows.append({
            "model": m["id"],
            "name": m["name"],
            "context": m.get("context_length"),
            "$ in /M": round(float(price.get("prompt") or 0) * 1e6, 3),
            "$ out /M": round(float(price.get("completion") or 0) * 1e6, 3),
            "tools": "tools" in params,
            "structured": "structured_outputs" in params,
            "free": m["id"].endswith(":free"),
        })
    df = pd.DataFrame(rows)
    return df[df["$ out /M"] >= 0]  # drops routers like openrouter/auto that have no fixed price


def find(df, text="", tools=False, max_out_price=None):
    """Filter a model table: text matches the model or name, tools=True keeps tool-calling models."""
    out = df
    if text:
        out = out[out["model"].str.contains(text, case=False) | out["name"].str.contains(text, case=False)]
    if tools and "tools" in out:
        out = out[out["tools"]]
    if max_out_price is not None:
        out = out[out["$ out /M"] <= max_out_price]
    return out.sort_values("$ out /M") if "$ out /M" in out else out


or_models = openrouter_models()
print(f"{len(or_models)} models on OpenRouter; {or_models['tools'].sum()} can call tools; {or_models['free'].sum()} are free")

In [ ]:
# Cheap models that can call tools (good for agents) - change the filters to explore
find(or_models, tools=True, max_out_price=1.0)

In [ ]:
# Some families to browse: try "anthropic/", "google/", "x-ai/", "moonshotai/", "qwen/", "deepseek/", "mistralai/"
find(or_models, "openai/gpt-5")

In [ ]:
# Free models (rate-limited, and your prompts may be used for training - check each model's page)
or_models[or_models["free"]]

In [ ]:
headers = {"Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}"}
credits = requests.get("https://openrouter.ai/api/v1/credits", headers=headers, timeout=30).json()["data"]
key_info = requests.get("https://openrouter.ai/api/v1/key", headers=headers, timeout=30).json()["data"]

left = credits["total_credits"] - credits["total_usage"]
print(f"OpenRouter credit: ${credits['total_credits']:.2f} bought, ${credits['total_usage']:.4f} used, ${left:.2f} left")
if key_info.get("limit") is not None:
    print(f"This key's spending limit: ${key_info['limit']} (${key_info['limit_remaining']} remaining)")
print("Per-request costs: https://openrouter.ai/activity")

## Gemini (Google)

Called with my `GEMINI_API_KEY` (my `GOOGLE_API_KEY` is the same key). Pricing: https://ai.google.dev/gemini-api/docs/pricing

In [ ]:
from google import genai

gemini = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
gemini_models = pd.DataFrame([
    {
        "model": m.name.removeprefix("models/"),
        "name": m.display_name,
        "input tokens": m.input_token_limit,
        "output tokens": m.output_token_limit,
        "thinking": m.thinking,
        "chat": "generateContent" in (m.supported_actions or []),
        "embeddings": "embedContent" in (m.supported_actions or []),
    }
    for m in gemini.models.list()
])
print(f"{len(gemini_models)} Gemini models; chat models below, embedding models: "
      f"{list(gemini_models[gemini_models['embeddings']]['model'])}")
gemini_models[gemini_models["chat"]].drop(columns=["chat", "embeddings"])

## DeepSeek

Very cheap. Its own API can't do structured outputs (`output_type=`, `with_structured_output`); for those, use
`PROVIDER = "openrouter"` with `MEDIUM = "deepseek/deepseek-v4-flash"`. Pricing: https://api-docs.deepseek.com/quick_start/pricing

In [ ]:
deepseek = client("deepseek")
print("Models:", [m.id for m in deepseek.models.list()])

balance = requests.get("https://api.deepseek.com/user/balance",
                       headers={"Authorization": f"Bearer {os.environ['DEEPSEEK_API_KEY']}"}, timeout=30).json()
for info in balance["balance_infos"]:
    print(f"Balance: {info['total_balance']} {info['currency']}")

## Anthropic (Claude)

The labs reach Claude through Anthropic's OpenAI-compatible endpoint, or through native Claude classes where the
framework has one (LangChain, Pydantic AI, Strands, Agno). Check your balance on the Billing page of the Claude Console
(platform.claude.com).

In [ ]:
import anthropic

claude = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY
pd.DataFrame([
    {"model": m.id, "name": m.display_name, "context": m.max_input_tokens, "max output": m.max_tokens,
     "released": m.created_at.date()}
    for m in claude.models.list()
])

## Ollama (local, free)

Models on this Mac. The Ollama app must be running (menu bar icon). A model loads into memory when you first call it
and unloads after 5 idle minutes. Download more with `ollama pull <name>`; remove with `ollama rm <name>`.

In [ ]:
try:
    tags = requests.get("http://localhost:11434/api/tags", timeout=5).json()["models"]
    loaded = {m["name"] for m in requests.get("http://localhost:11434/api/ps", timeout=5).json()["models"]}
    rows = []
    for m in tags:
        info = requests.post("http://localhost:11434/api/show", json={"model": m["name"]}, timeout=10).json()
        rows.append({
            "model": m["name"],
            "size GB": round(m["size"] / 1e9, 1),
            "parameters": info.get("details", {}).get("parameter_size"),
            "capabilities": ", ".join(info.get("capabilities", [])),
            "loaded now": m["name"] in loaded,
        })
    display(pd.DataFrame(rows))
except requests.ConnectionError:
    print("Ollama isn't running - open the Ollama app, then re-run this cell")

## Try any model

`try_model(provider, model)` sends one prompt and shows the reply and how long it took. Each call costs a fraction of a cent
(Ollama is free).

In [ ]:
def try_model(provider, model, prompt="In one sentence, what makes an AI agent different from a chatbot?"):
    start = time.perf_counter()
    try:
        reply = client(provider).chat.completions.create(model=model, messages=[{"role": "user", "content": prompt}])
        text = reply.choices[0].message.content.strip()
    except Exception as e:
        text = f"FAILED: {type(e).__name__}: {e}"
    print(f"{provider} / {model}  ({time.perf_counter() - start:.1f}s)\n{text}\n")


try_model("openrouter", "moonshotai/kimi-k2.6")

In [ ]:
# Compare the MEDIUM default of every provider on the same prompt
for provider, (small, medium, large) in MODELS.items():
    try_model(provider, medium)

## Local embeddings

`qwen3-embedding:0.6b` turns text into vectors on this Mac, for retrieval (RAG) experiments. It's a stand-in for OpenAI's
`text-embedding-3-small`. Gemini's `gemini-embedding-001` is a cloud option with `client("gemini")`.

In [ ]:
import math

texts = ["How do I give my agent web search?", "Add a search tool to the agent.", "The recipe needs two eggs."]
vectors = [d.embedding for d in client("ollama").embeddings.create(model="qwen3-embedding:0.6b", input=texts).data]


def cosine(a, b):
    return sum(x * y for x, y in zip(a, b)) / (math.sqrt(sum(x * x for x in a)) * math.sqrt(sum(y * y for y in b)))


print(f"{len(vectors[0])} numbers per text")
for text, vector in zip(texts[1:], vectors[1:]):
    print(f"{cosine(vectors[0], vector):.3f}  similarity to: {text}")